## Original SCB agent prompt provided in the paper
### Before the modular impl. agents are implemented, use this 
Specify additionally that follow the design in `current_design.json`

In [1]:
from prompts.agent_prompts.scb import get_original_scb_prompt

# Pass the current checkpoint number to be implemented 
print(get_original_scb_prompt(checkpoint_number=2))


Implement a program that 100% solves the specification.
That is all you need to do.

Keep using the same virtual environment you started with,
update 'requirements.txt' with any new dependencies you need.

You are working on the following issue: checkpoint_2.md
Implement your solution in implementation/ folder.
Extend your solution based on previous_implementation/.



## Reader agent (Unnecessary because we have metrics?)
Its purpose is to intiialise the `current_design.json` for the loop, and does not output anything. 

In [2]:
from prompts.agent_prompts.reader import get_reader_prompt

print(get_reader_prompt(3))


You are a senior software engineer that analyses modules in a software project.

Your job is to understand the following directory: 
Project root: agent_workspace
Directory: checkpoint_2/
Dependency graph: checkpoint_2_graph.json

Identify the existing modules in the codebase, using the dependency graph as reference. For each module, identify its responsibility. Do not propose new modules.

Write a JSON object to `current_design.json`, that describes each module using the following schema. If there are no modules, write an empty JSON array.
{
  "type": "array",
  "items": {
    "type": "object",
    "properties": {
      "module_name": {
        "description": "The name of the module.",
        "type": "string"
      },
      "module_path": {
        "description": "The directory path of the module.",
        "type": "string"
      },
      "responsibility": {
        "description": "A sentence describing the single responsibility of this module.",
        "type": "string"
      }
   

## Analyzer agent

In [3]:
from prompts.agent_prompts.analyzer import get_analyzer_prompt

# The actual DPy results should be filtered for specific things we concern only. 
# For this TEST example, a path is provided instead. DO NOT do this in your final submission 
print(get_analyzer_prompt(has_implementation=False))


You are a senior software code quality analyst. 
Your job is to evaluate the following modular design including kept, changed or new modules.
Project root: agent_workspace
Design: `current_design.json`
Dependency graph: `current_deps_graph.json` 

You work with a OBSERVE - SUPPORT - SCORE cycle and you should not skip steps when performing your evaluation: 
    
    OBSERVE: Observe the design, and state antipatterns or maintainability issues that you have discovered. 
    Example: This module / file seems to mixes two separate, equally complex logic together. 

    SUPPORT: Only after the codebase observation then consider the code smells identified in `current_metrics.json`. You may use them to support your claim and the smells are not to be eliminated completely, and sometimes smells are not available for certain code smells, such as violation of single responsibilities or duplicated logic. 
    Example: It is found that there is a high LCOM in the aforementioned class, and we can 

## Decomposer Agent

In [4]:
from prompts.agent_prompts.decomposer import get_decomposer_prompt

# Extract only the improvements part of the analyzer output 
print(get_decomposer_prompt(1))


You are a senior software engineer that specialises in modular software design.

You are working on the following issue:
Project root: agent_workspace
Issue path: checkpoint_1.md


If a design is provided in `current_design.json` with the dependency graph in `current_deps_graph.json`, prioritise reusing existing modules instead of creating a new module where possible.
If a list of improvement suggestions for the current design is provided in `current_analyzer_result.json`, please consider accepting or rejecting them based on: 
- Whether the suggestion lead to reduced future effort when adding new features.
- Whether the nature of the problem and the size of the file justifies the complexity without the refactoring. 
- Whether the suggestion conflict with issue requirements.


Propose a modular design that achieves the goal specified in the issue when integrated together. 
You should follow best code practices, including: 
- A module should only expose the minimum amount of knowledge i

## Analyzer on the implementation

In [5]:
from prompts.agent_prompts.analyzer import get_analyzer_prompt

print(get_analyzer_prompt(has_implementation=True))


You are a senior software code quality analyst. 
Your job is to evaluate the following implementation including kept, changed or new modules.
Project root: agent_workspace
Implementation: implementation/
Dependency graph: `current_deps_graph.json`, which is modified from `original_deps_graph.json`. 

You work with a OBSERVE - SUPPORT - SCORE cycle and you should not skip steps when performing your evaluation: 
    
    OBSERVE: Observe the code implementation, and state antipatterns or maintainability issues that you have discovered. 
    Example: This module / file seems to mixes two separate, equally complex logic together. 

    SUPPORT: Only after the codebase observation then consider the code smells identified in `current_metrics.json`. You may use them to support your claim and the smells are not to be eliminated completely, and sometimes smells are not available for certain code smells, such as violation of single responsibilities or duplicated logic. 
    Example: It is found

In [6]:
ANALYZER_OUTPUT_2 = {
    "result": "pass",
    "improvements": [
      {
        "module_name": "pipeline.ast_nodes",
        "smell": "Module conflates language-level AST node types with application-level caching configuration structures, reducing cohesion and coupling the language frontend to the caching subsystem.",
        "improvement_instruction": "Extract TtlConfig, CacheKeyConfig, CacheConfig, and GlobalCacheConfig into a dedicated pipeline.cache_config module. pipeline.ast_nodes should retain only constructs that represent parsed language elements (TaskDef, ParamDef, and token-adjacent types). Update imports in pipeline.cache_key, pipeline.cache_manager, pipeline.executor, and pipeline.main accordingly."
      },
      {
        "module_name": "pipeline.expr_parser",
        "smell": "parse_block returns List[Any], erasing all type information at the parser/evaluator boundary despite typed AST node dataclasses existing in pipeline.ast_nodes.",
        "improvement_instruction": "Define a StmtNode union type or a common base dataclass in pipeline.ast_nodes that covers all statement node variants (IfStmt, ForStmt, WhileStmt, AssignStmt, ReturnStmt, etc.). Change ExprParser.parse_block to return List[StmtNode] so that static type checking is preserved across the parse/evaluate boundary."
      },
      {
        "module_name": "pipeline.evaluator",
        "smell": "eval_block accepts raw List[Token] and internally invokes ExprParser, conflating token parsing with expression evaluation and violating the established lex-parse-evaluate layering.",
        "improvement_instruction": "Remove the token-to-AST parsing step from eval_block. Change its signature to accept List[StmtNode] (a pre-parsed AST). Callers such as Executor should invoke ExprParser.parse_block explicitly before calling eval_block, keeping the two phases separately testable and aligned with the pipeline.expr_parser/pipeline.evaluator module boundary."
      },
      {
        "module_name": "pipeline.cache_manager",
        "smell": "store() accepts both the precomputed cache_key and the raw inputs (task_def, params, workspace) from which the key was derived, producing a redundant and inconsistent method signature.",
        "improvement_instruction": "Simplify store() to accept only task_def (for cache location resolution), cache_key, and job_result. Remove the redundant params and workspace parameters; since cache_key is already computed by a prior check() call, only the cache directory (derivable from task_def.cache.location) is needed to persist the entry."
      },
      {
        "module_name": "pipeline.cache_store",
        "smell": "The exists() method is fully subsumed by load() returning None, unnecessarily widening the public interface and enabling TOCTOU access patterns.",
        "improvement_instruction": "Remove the exists() method from cache_store's public interface. Update all callers in pipeline.cache_manager to use load() and branch on the None return value, eliminating the separate existence check."
      }
    ]
  }

# Modular coder agent

In [7]:
from prompts.agent_prompts.coders.modular_coder import get_modular_coder_prompt

print(
    get_modular_coder_prompt(2,['pipeline/requires_executor.py', 'pipeline/success_evaluator.py'])
)


You are working on the following issue: checkpoint_2.md
Implement your solution in implementation/ folder.
Extend your solution based on previous_implementation/.

Keep using the same virtual environment you started with,
update 'requirements.txt' with any new dependencies you need.

Ensure good coding practices by:  
- Avoid functions that are too complex with too much nested if/else statements.
- Avoid the use of magic numbers when their meanings are not obvious.
- Do not access the private elements of another class.

A dependency graph will later be generated by pointing to the entrypoint file. 
Use flat file imports and do not add an __init__.py file inside the implementation/ folder. 

Your job is to implement the following modules. Follow the design specified in `current_design.json` and the module dependencies in `current_deps_graph.json`, import existing modules where possible.
- pipeline/requires_executor.py
- pipeline/success_evaluator.py.



In [8]:
from prompts.agent_prompts.coders.modular_coder import get_all_at_once_coder_prompt
print(
    get_all_at_once_coder_prompt(2)
)

ImportError: cannot import name 'get_all_at_once_coder_prompt' from 'prompts.agent_prompts.coders.modular_coder' (/home/bernardmoy/modular-SWE/prompts/agent_prompts/coders/modular_coder.py)

In [ ]:
from prompts.agent_prompts.coders.modular_coder import get_no_design_coder_prompt
print(
    get_no_design_coder_prompt(2)
)


You are working on the following issue: checkpoint_2.md
Implement your solution in implementation/ folder.
Extend your solution based on previous_implementation/.

Keep using the same virtual environment you started with,
update 'requirements.txt' with any new dependencies you need.

Ensure good coding practices by:  
- Avoid functions that are too complex with too much nested if/else statements.
- Avoid the use of magic numbers when their meanings are not obvious.
- Do not access the private elements of another class.

A dependency graph will later be generated by pointing to the entrypoint file. 
Use flat file imports and do not add an __init__.py file inside the implementation/ folder. 

Implement a program that 100% solves the specification.
That is all you need to do.



## Refactor coder agent

In [ ]:
from prompts.agent_prompts.coders.refactor_coder import get_refactor_coder_prompt
print(
    get_refactor_coder_prompt(2)
)


You are a senior software engineer that specialises in modular software design.

You are working on the following issue:
Project root: agent_workspace
Issue path: checkpoint_2.md
Implementation folder: implementation/

If a list of improvement suggestions for the current design is provided in `current_analyzer_result.json`, please consider accepting or rejecting them based on: 
- Whether the suggestion lead to reduced future effort when adding new features.
- Whether the nature of the problem justifies the current complexity without the refactoring.
- Whether the suggestion conflict with issue requirements.

Your task is to perform refactoring on the implementation code while preserving its functionality based on the improvement suggestions and your evaluation. 
Additionally, create or add to the JSON object in `current_rejected_improvements.json` including each improvement suggestion that was rejected, using the following schema. Do NOT remove existing rejected suggestions unless nec